**Import tableau final**

In [1]:
!pip install --upgrade google-cloud-bigquery
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery

client = bigquery.Client(project="projet-riskhorizon-2276")


In [2]:
query = """
SELECT *
FROM `projet-riskhorizon-2276.3_Mart.tableau_final_table`
"""

df = client.query(query).to_dataframe()

df.head()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,dette_totale_ext,ratio_dette_credit_ext,nb_credits_en_retard_ext,retard_max_ext,montant_total_retard_ext,nb_prolongations_ext,nb_types_credit_ext,nb_demandes_passees_int,nb_refus_passes_int,montant_total_emprunte_passe_int
0,355557,0,CASH_LOANS,F,False,True,0,27000.0,45000.0,4450.5,...,122050.98,0.474226,0,0,0.0,0,2,1,0,75312.0
1,211865,0,CASH_LOANS,F,False,True,0,27000.0,45000.0,4383.0,...,637623.00,0.421575,0,0,0.0,0,2,6,0,660501.0
2,364118,0,CASH_LOANS,F,False,True,0,29250.0,45000.0,4855.5,...,NaN,NaN,<NA>,<NA>,NaN,<NA>,<NA>,5,2,207724.5
3,143626,1,CASH_LOANS,F,False,True,0,31500.0,45000.0,5076.0,...,NaN,NaN,<NA>,<NA>,NaN,<NA>,<NA>,1,0,32503.5
4,372569,0,CASH_LOANS,F,False,False,0,31500.0,45000.0,4450.5,...,29817.00,0.047083,0,0,0.0,0,2,2,0,207616.5


**WORKFLOW COMPLET**

In [3]:
# [ ÉTAPE 1 : COLLECTE & CHARGEMENT ]
#    SUPPRESSION COLONNES
#    df = pd.read_csv(...) : Import de vos données brutes (TARGET, SK_ID_CURR, variables).
#         │
# [ ÉTAPE 2 : SÉPARATION DES CIBLES & IDENTIFIANTS ]
#    Extraction de 'y' (TARGET) et 'client_id' (SK_ID_CURR).
#    Création de 'X' en supprimant ces deux colonnes du dataset de travail.
#         │
# [ ÉTAPE 3 : ENCODAGE DES VARIABLES CATÉGORIELLES ]
#    X_encoded = pd.get_dummies(X, drop_first=True) : Transformation du texte en chiffres (0 ou 1).
#    Sauvegarde de 'feature_names' pour l'analyse finale.
#         │
# [ ÉTAPE 4 : LE SPLIT INITIAL (Le Coffre-Fort) ]
#    train_test_split(..., test_size=0.2, stratify=y) : On isole 20% des données (X_test, y_test).
#    Elles restent totalement secrètes et intouchées jusqu'à la fin.
#         │
# [ ÉTAPE 5 : DÉFINITION DU SQUELETTE (Le Pipeline) ]
#    Déclaration de la chaîne de montage sans données :
#      SimpleImputer (médiane) ➔ StandardScaler ➔ LogisticRegression(class_weight="balanced").
#         │
# [ ÉTAPE 6 : VALIDATION CROISÉE (L'Évaluation de la stratégie) ]
#    cross_val_score(model_pipeline, X_train, y_train, cv=5) :
#      Simulation de 5 entraînements / tests internes pour calculer l'AUC Moyen et l'Écart-type (.std()).
#    Objectif : Valider que le modèle est stable et ne fait pas d'overfitting.
#         │
# [ ÉTAPE 7 : LE FIT FINAL (L'Entraînement définitif) ]
#    model_pipeline.fit(X_train, y_train) :
#      Le pipeline apprend DEFINITIVEMENT les médianes, les moyennes et les coefficients sur 100% du train.
#         │
# [ ÉTAPE 8 : LE TEST RÉEL (L'Évaluation finale) ]
#    y_pred_test = model_pipeline.predict_proba(X_test)[:, 1]
#    Calcul de l'AUC Test Final : On vérifie si la performance se maintient sur les données secrètes.
#         │
# [ ÉTAPE 9 : EXTRACTION DE L'IMPORTANCE DES VARIABLES ]
#    coefficients = model_pipeline.named_steps['classifier'].coef_[0]
#    Analyse et tri des variables les plus influentes via le graphique Plotly.
#         │
# [ ÉTAPE 10 : Matrice de confusion ]
#    cm = confusion_matrix(y_test, y_pred_class)
#    cm_normalized = confusion_matrix(y_test, y_pred_class, normalize='true') # Taux par ligne
#    Graphique plotly


**ETAPE 1 à 8**

***Categorisation manuelle de la colonne NAME_EDUCATION_TYPE***

In [4]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import roc_auc_score
from sklearn.metrics import roc_auc_score, confusion_matrix

#CATEGORIE EDUCATION
# Plus le niveau est élevé, plus la note augmente
education_mapping = {
    'LOWER_SECONDARY': 0,
    'SECONDARY_/_SECONDARY_SPECIAL': 1,
    'INCOMPLETE_HIGHER': 2,
    'HIGHER_EDUCATION': 3
}

df['EDUCATION_RANK'] = df['NAME_EDUCATION_TYPE'].map(education_mapping)

# Filet de sécurité : si une valeur est manquante ou mal orthographiée,
df['EDUCATION_RANK'] = df['EDUCATION_RANK'].fillna(1)


In [5]:
# import plotly.express as px

# fig = px.histogram(df, x='EDUCATION_RANK', title='Distribution of EDUCATION_RANK')
# fig.show()

***Exclusion des variables à faible coefficient***

In [6]:
# On supprime la colonne de texte d'origine pour ne pas qu'elle soit encodée par get_dummies
features_to_drop = [
    "NAME_EDUCATION_TYPE",
    "nb_credits_ext",
    "AVG_CREDIT_SCORE",
    "total_credit_ext",
    "nb_prolongations_ext",
    "retard_max_ext",
    "montant_total_retard_ext",
    "AMT_ANNUITY",
    "Reste_a_vivre",
    "Reste_a_vivre_par_personne",
    "CNT_CHILDREN",
    "AMT_INCOME_TOTAL",
    "AGE",
    "dette_totale_ext",
    "nb_credits_en_retard_ext",
    "NAME_FAMILY_STATUS",
    "NAME_CONTRACT_TYPE",
    "FLAG_OWN_REALTY"
]

organization_cols = [c for c in df.columns if "ORGANIZATION_TYPE" in c]
features_to_drop.extend(organization_cols)

y = df["TARGET"]
X = df.drop(columns=["TARGET", "SK_ID_CURR"] + [f for f in features_to_drop if f in df.columns])


In [7]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 307497 entries, 0 to 307496
Data columns (total 20 columns):
 #   Column                            Non-Null Count   Dtype  
---  ------                            --------------   -----  
 0   CODE_GENDER                       307497 non-null  object 
 1   FLAG_OWN_CAR                      307497 non-null  boolean
 2   AMT_CREDIT                        307497 non-null  float64
 3   AMT_GOODS_PRICE                   307497 non-null  float64
 4   NAME_INCOME_TYPE                  307497 non-null  object 
 5   EXT_SOURCE_1                      134124 non-null  float64
 6   EXT_SOURCE_2                      306837 non-null  float64
 7   EXT_SOURCE_3                      246534 non-null  float64
 8   YEARS_EMPLOYED                    307497 non-null  float64
 9   tx_endettement                    307497 non-null  float64
 10  APPORT_ESTIME                     307497 non-null  float64
 11  RATIO_PRET                        307497 non-null  f

In [8]:
#Encodage (get_dummies) avant le split
X_encoded = pd.get_dummies(X, drop_first=True)

# Sauvegarde des noms de variables pour l'interprétation
feature_names = X_encoded.columns

# Split initial
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

#Création du Pipeline (preparation usinage)
preprocessing = Pipeline([
    ('imputer', SimpleImputer(strategy="median")),#=> remplace valeur manquante par mediane
    ('scaler', StandardScaler())# => met à l'échelle toutes les données
])

model_pipeline = Pipeline([
    ('preprocessor', preprocessing),#=> valeurs manquantes + standardisation
    ('classifier', LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))#=> algorithme de Régression Logistique
])

#class_weight="balanced"=>permet d'équilibrer la répartion des target (trop peu de 1)
# TARGET
# 0   91.9268
# 1    8.0732


#Validation Croisée (StratifiedKFold)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42) #=> n_splits=5 : doit découper les données en 5 morceaux (les folds) pour
cv_scores = cross_val_score(model_pipeline, X_train, y_train, cv=cv, scoring='roc_auc')

print("VALIDATION CROISÉE")
print(f"Scores AUC par fold : {cv_scores}")
print(f"Score AUC Moyen : {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")
print("-" * 30)

# Entraînement Final & Évaluation
model_pipeline.fit(X_train, y_train)

y_pred_test = model_pipeline.predict_proba(X_test)[:, 1]
print(f"AUC Test Final : {roc_auc_score(y_test, y_pred_test):.4f}\n")



VALIDATION CROISÉE
Scores AUC par fold : [0.74782651 0.74728479 0.75142024 0.74175986 0.74689786]
Score AUC Moyen : 0.7470 (+/- 0.0062)
------------------------------
AUC Test Final : 0.7437



In [10]:
from sklearn.metrics import accuracy_score, recall_score, precision_score

# 1. On récupère les prédictions globales sur le jeu de test
# (Remplacez 'log_reg_model' par le nom de votre variable de modèle)
y_pred_all = model_pipeline.predict(X_test)

# 2. On crée un masque pour isoler uniquement les hommes ('M') dans le jeu de test
# Note : df.loc[X_test.index] permet de retrouver la colonne d'origine dans votre DataFrame de départ
masque_hommes = (df.loc[X_test.index, "CODE_GENDER"] == "M")

# 3. On filtre les vraies étiquettes (y) et les prédictions pour n'en garder que les hommes
y_test_hommes = y_test[masque_hommes]
y_pred_hommes = y_pred_all[masque_hommes]

# 4. Calcul des métriques demandées
accuracy_h = accuracy_score(y_test_hommes, y_pred_hommes)
recall_h = recall_score(y_test_hommes, y_pred_hommes)
precision_h = precision_score(y_test_hommes, y_pred_hommes)

# 5. Affichage des résultats
print("--- PERFORMANCES MODÈLE : HOMMES UNIQUEMENT (CODE_GENDER = M) ---")
print(f"Nombre d'hommes dans le jeu de test : {sum(masque_hommes)}")
print(f"Accuracy  : {accuracy_h:.4f}")
print(f"Recall    : {recall_h:.4f}")
print(f"Precision : {precision_h:.4f}")

--- PERFORMANCES MODÈLE : HOMMES UNIQUEMENT (CODE_GENDER = M) ---
Nombre d'hommes dans le jeu de test : 20931
Accuracy  : 0.5998
Recall    : 0.7553
Precision : 0.1691


In [ ]:
print(y.value_counts(normalize=True) * 100)

TARGET
0    91.926751
1     8.073249
Name: proportion, dtype: Float64


**ETAPE 9**

In [ ]:
# On récupère les coefficients du modèle linéaire
coefficients = model_pipeline.named_steps['classifier'].coef_[0]

# Création d'un DataFrame pour analyser les poids
importance_df = pd.DataFrame({
    'Variable': feature_names,
    'Coefficient': coefficients,
    'Abs_Coefficient': np.abs(coefficients) # Pour trier par impact absolu
}).sort_values(by='Abs_Coefficient', ascending=False)

# Visualisation des 20 variables les plus influentes
fig_importance = px.bar(
    importance_df.head(20),
    x='Coefficient',
    y='Variable',
    orientation='h',
    title='Top 20 des variables les plus influentes (Coefficients Régression Logistique)',
    labels={'Coefficient': 'Poids du coefficient (Positif = Risque de défaut, Négatif = Sûr)'},
    color='Coefficient',
    color_continuous_scale=px.colors.diverging.RdYlGn[::-1] # Rouge pour risque, Vert pour sûr
)
fig_importance.update_layout(yaxis={'categoryorder':'total ascending'})
fig_importance.show()

In [ ]:
new_importance_df = importance_df.copy()
# limiter 4 decimals après virgule
new_importance_df['Coefficient'] = new_importance_df['Coefficient'].apply(lambda x: '{:,.4f}'.format(x))
new_importance_df['Abs_Coefficient'] = new_importance_df['Abs_Coefficient'].apply(lambda x: '{:,.4f}'.format(x))

print(new_importance_df.to_string(index=False))

                        Variable Coefficient Abs_Coefficient
                    EXT_SOURCE_2     -0.4269          0.4269
                    EXT_SOURCE_3     -0.4048          0.4048
             nb_refus_passes_int      0.3092          0.3092
         nb_demandes_passees_int     -0.2910          0.2910
                    EXT_SOURCE_1     -0.2009          0.2009
                   CODE_GENDER_M      0.1840          0.1840
      NAME_INCOME_TYPE_PENSIONER     -0.1835          0.1835
                  YEARS_EMPLOYED     -0.1832          0.1832
                  EDUCATION_RANK     -0.1588          0.1588
                  tx_endettement      0.1423          0.1423
                      RATIO_PRET      0.1423          0.1423
montant_total_emprunte_passe_int      0.1411          0.1411
                    FLAG_OWN_CAR     -0.1323          0.1323
           nb_credits_actifs_ext      0.1202          0.1202
         nb_credits_clotures_ext     -0.0773          0.0773
          ratio_dette_cr

In [ ]:
from google.cloud import bigquery

# Define your BigQuery project, dataset, and table name
project_id = "projet-riskhorizon-2276"
dataset_id = "3_Mart"
table_id = "coeff_regression_variables"

destination_table = f"{project_id}.{dataset_id}.{table_id}"

# Export df_final to BigQuery
new_importance_df.to_gbq(
    destination_table,
    project_id=project_id,
    if_exists='replace'
)

print(f"DataFrame 'df_final' successfully exported to BigQuery table: {destination_table}")

/tmp/ipykernel_422/2106634183.py:11: FutureWarning:

to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq

100%|██████████| 1/1 [00:00<00:00, 5698.78it/s]

DataFrame 'df_final' successfully exported to BigQuery table: projet-riskhorizon-2276.3_Mart.coeff_regression_variables


**Etape 10 : Matrice de confusion**

In [ ]:
# conversion des probabilités en classes (Seuil par défaut à 0.5)
y_pred_class = (y_pred_test >= 0.5).astype(int)

# calcul de la matrice brute et normalisée
cm = confusion_matrix(y_test, y_pred_class)
cm_normalized = confusion_matrix(y_test, y_pred_class, normalize='true') # Taux par ligne

# création du graphique Plotly
labels_etats = ['Non-Défaut (0)', 'Défaut (1)']

fig_cm = px.imshow(
    cm_normalized,
    text_auto=".2%", # Affiche les taux en pourcentage
    x=labels_etats,
    y=labels_etats,
    labels=dict(x="État PRÉDIT par le modèle", y="État RÉEL du client", color="Taux"),
    title="Matrice de Confusion (Transition Réel vs Prédit)",
    color_continuous_scale="Blues"
)

fig_cm.show()

***Nouvelle table avec coefficient par client :***

In [ ]:
# 1. Calcule directement les probabilités sur X_encoded (le pipeline gère le scaling en interne)
proba_full = model_pipeline.predict_proba(X_encoded)[:, 1]

# 2. On calcule le score de confiance (plus il est proche de 1, plus le client est sûr)
score_full = 1 - proba_full

# 3. Création de ton DataFrame final avec les résultats
df_final = df.copy()
df_final["proba_defaut"] = proba_full
df_final["score"] = score_full

In [ ]:
df_final.head()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,retard_max_ext,montant_total_retard_ext,nb_prolongations_ext,nb_types_credit_ext,nb_demandes_passees_int,nb_refus_passes_int,montant_total_emprunte_passe_int,EDUCATION_RANK,proba_defaut,score
0,355557,0,CASH_LOANS,F,False,True,0,27000.0,45000.0,4450.5,...,0,0.0,0,2,1,0,75312.0,1.0,0.599416,0.400584
1,211865,0,CASH_LOANS,F,False,True,0,27000.0,45000.0,4383.0,...,0,0.0,0,2,6,0,660501.0,1.0,0.131708,0.868292
2,364118,0,CASH_LOANS,F,False,True,0,29250.0,45000.0,4855.5,...,<NA>,NaN,<NA>,<NA>,5,2,207724.5,1.0,0.440325,0.559675
3,143626,1,CASH_LOANS,F,False,True,0,31500.0,45000.0,5076.0,...,<NA>,NaN,<NA>,<NA>,1,0,32503.5,1.0,0.446742,0.553258
4,372569,0,CASH_LOANS,F,False,False,0,31500.0,45000.0,4450.5,...,0,0.0,0,2,2,0,207616.5,3.0,0.077283,0.922717


In [ ]:
from google.cloud import bigquery

# Define your BigQuery project, dataset, and table name
project_id = "projet-riskhorizon-2276"
dataset_id = "3_Mart"
table_id = "tableau_final_LogisticRegression_v3"

destination_table = f"{project_id}.{dataset_id}.{table_id}"

# Export df_final to BigQueryS
df_final.to_gbq(
    destination_table,
    project_id=project_id,
    if_exists='replace'
)

print(f"DataFrame 'df_final' successfully exported to BigQuery table: {destination_table}")

/tmp/ipykernel_422/682671322.py:11: FutureWarning:

to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq

100%|██████████| 1/1 [00:00<00:00, 8612.53it/s]

DataFrame 'df_final' successfully exported to BigQuery table: projet-riskhorizon-2276.3_Mart.tableau_final_LogisticRegression_v3
